# vbOCP — Test 1: pipeline FOM

Notebook orchestratore: le celle di codice lanciano solo comandi bash (`!`) verso gli script della repo — nessuna logica del modello qui dentro, solo chiamate. Vale il pattern pensato per Colab/cluster: stesso codice, stesso comando, cambia solo dove lo lanci.

**Contenuto:**
1. Setup (path della repo)
2. Solve singolo + plot dello stato/aggiunto per una terna di parametri
3. Generazione di un dataset di snapshot (campionamento casuale di mu1, mu2, mu_u)

## 1. Setup

In [ ]:
import os

# si sposta nella root della repo (dove c'e' src/), indipendentemente da dove parte il kernel
while not os.path.isdir('src') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')
assert os.path.isdir('src'), "src/ non trovata: verifica dove e' montata la repo vbOCP"

REPO_ROOT     = os.getcwd()
CONFIG_PATH   = os.path.join('configs', 'test1.yaml')
DATA_DIR      = os.path.join('data')
SNAPSHOTS_DIR = os.path.join('data', 'snapshots')
OUTPUT_DIR    = os.path.join('notebooks', 'output')
PLOT_PATH     = os.path.join(OUTPUT_DIR, 'fom_solution.png')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SNAPSHOTS_DIR, exist_ok=True)

print('REPO_ROOT    :', REPO_ROOT)
print('CONFIG_PATH  :', CONFIG_PATH)
print('DATA_DIR     :', DATA_DIR)
print('SNAPSHOTS_DIR:', SNAPSHOTS_DIR)
print('PLOT_PATH    :', PLOT_PATH)

## 2. Solve singolo + plot

Risolve il FOM per una terna di parametri e salva un confronto visivo di stato `y` e aggiunto `p`. Modifica `--mu1`/`--mu2`/`--mu_u` per esplorare altri punti dello spazio parametrico.

In [ ]:
!python -m src.full_order.run_single_solve \
    --config {CONFIG_PATH} \
    --mu1 12 --mu2 2.5 --mu_u 0.99 \
    --plot --output {PLOT_PATH}

In [ ]:
from IPython.display import Image
Image(PLOT_PATH)

## 3. Generazione snapshot

Campiona `mu1`, `mu2`, `mu_u` uniformemente sui range definiti in `configs/test1.yaml` e risolve il FOM per ciascun campione. L'assembly indipendente da mu viene fatto una sola volta; il ciclo aggiorna solo la matrice di controllo e il solve.

Output: un unico file `.npz` con `mu1`, `mu2`, `mu_u`, e le matrici `Y`, `P`, `U` (una colonna per campione).

In [ ]:
N_SAMPLES = 5  # test veloce - alzare a 300 per un dataset POD vero
SNAPSHOTS_PATH = os.path.join(SNAPSHOTS_DIR, f'test1_{N_SAMPLES}.npz')

!python -m src.full_order.generate_snapshots \
    --config {CONFIG_PATH} \
    --n-samples {N_SAMPLES} \
    --output {SNAPSHOTS_PATH}

In [ ]:
import numpy as np

data = np.load(SNAPSHOTS_PATH)
print('Chiavi salvate:', list(data.keys()))
print('Y shape:', data['Y'].shape)
print('mu1 campionati:', data['mu1'])